# Program 2 — custom objects + COCO objects (YOLO11n-seg)

Takes a finished Program 1 version, adds the COCO objects you choose, cross-labels both ways
(you approve by hand), and trains a fresh model from `yolo11n-seg.pt`.

Uses the same Drive folder as Program 1. Results go to `MyDrive/yolo/full/v1_16objects_2026-11-08/`.

**Run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.
The first run stops to show your custom image counts so you can choose `PER_CLASS_TRAIN`.

In [ ]:
# ============================== SETTINGS ==============================
ROOT = "/content/drive/MyDrive/yolo"
CUSTOM_VERSION = "latest"    # Program 1 result to use: "latest" finished one, or e.g. "v1_2objects_2026-11-08"

COCO_CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "train", "truck",
                "traffic light", "fire hydrant", "stop sign", "parking meter",
                "bench", "cat", "dog"]
PER_CLASS_TRAIN = None       # COCO train images per class. Leave None to see your custom counts first.
PER_CLASS_VAL = 150          # COCO val images per class

VERSION = "auto"
REVIEW = "manual"            # "manual" | "auto"
REVIEWED_VERSION = ""        # after reviewing, paste the version name here

CONF_COCO_ON_CUSTOM = 0.35   # adding COCO objects to your photos
CONF_CUSTOM_ON_COCO = 0.35   # adding your objects to COCO photos
MIN_AREA = 0.002

MODEL = "yolo11n-seg.pt"     # always trains from this
EPOCHS = 60
PATIENCE = 15
IMGSZ = 640
BATCH = 32
WORKERS = 2
TIMEZONE = "Asia/Singapore"
# =====================================================================

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q -U ultralytics
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# ---------------------------------------------------------------------------
# Shared helpers (this cell is identical in both notebooks)
# ---------------------------------------------------------------------------
import os, re, json, shutil, random, subprocess, datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LOCAL = Path("/content/work")        # fast local disk (lost on disconnect)
DATA = LOCAL / "data"                # dataset used for training


class StopHere(Exception):
    """Raised on purpose to stop 'Run all' and ask you to do something."""


def today():
    return datetime.datetime.now(ZoneInfo(TIMEZONE)).strftime("%Y-%m-%d")


def load_state(vdir):
    f = Path(vdir) / "state.json"
    return json.loads(f.read_text()) if f.exists() else {}


def save_state(vdir, st):
    Path(vdir).mkdir(parents=True, exist_ok=True)
    (Path(vdir) / "state.json").write_text(json.dumps(st, indent=1))


def version_dirs(parent):
    out = {}
    if Path(parent).is_dir():
        for d in Path(parent).iterdir():
            m = re.match(r"v(\d+)_", d.name)
            if d.is_dir() and m:
                out[int(m.group(1))] = d
    return out


def resolve_version(parent, version, n_objects):
    """'auto' = continue the newest unfinished version, otherwise start the next number."""
    found = version_dirs(parent)
    if version != "auto":
        return Path(parent) / version
    if found:
        newest = found[max(found)]
        if not load_state(newest).get("done"):
            return newest
    return Path(parent) / f"v{max(found, default=0) + 1}_{n_objects}objects_{today()}"


def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)


def zip_dirs(zip_path, members, cwd):
    """Store-only zip of `members` (paths relative to cwd) into zip_path on Drive."""
    zip_path = Path(zip_path)
    tmp = LOCAL / ("_tmp_" + zip_path.name)
    tmp.unlink(missing_ok=True)
    run(["zip", "-q", "-r", "-0", str(tmp)] + [str(m) for m in members], cwd=cwd)
    shutil.copy(tmp, zip_path)
    tmp.unlink()


def unzip(zip_path, dest):
    Path(dest).mkdir(parents=True, exist_ok=True)
    run(["unzip", "-q", "-o", str(zip_path), "-d", str(dest)])


def images_in(split=None, prefix=None):
    splits = [split] if split else ["train", "val", "test"]
    out = []
    for s in splits:
        d = DATA / s / "images"
        if d.exists():
            out += [p for p in sorted(d.iterdir())
                    if p.suffix.lower() in IMG_EXTS and (prefix is None or p.name.startswith(prefix))]
    return out


def write_yaml(names):
    lines = [f"path: {DATA}", "train: train/images", "val: val/images"]
    if images_in("test"):
        lines.append("test: test/images")
    lines += [f"nc: {len(names)}", "names:"] + [f"  {i}: {n}" for i, n in enumerate(names)]
    (DATA / "data.yaml").write_text("\n".join(lines) + "\n")


def polygon_area(xy):
    x, y = xy[:, 0], xy[:, 1]
    return 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def propose(weights, images, keep, remap, conf, tag, review_dir):
    """Run `weights` on `images`, keep classes `keep`, write proposals + previews."""
    import cv2
    from ultralytics import YOLO
    pdir = Path(review_dir) / "proposals" / tag
    vdir = Path(review_dir) / "previews" / tag
    pdir.mkdir(parents=True, exist_ok=True)
    vdir.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(weights))
    n = 0
    paths = [str(p) for p in images]
    for i in range(0, len(paths), 256):
        for r in model.predict(paths[i:i + 256], classes=keep, conf=conf, imgsz=IMGSZ,
                               retina_masks=True, stream=True, verbose=False, device=0):
            if r.masks is None or len(r.masks) == 0:
                continue
            lines = []
            for poly, c in zip(r.masks.xyn, r.boxes.cls.int().tolist()):
                if len(poly) < 3 or polygon_area(poly) < MIN_AREA:
                    continue
                pts = poly[::max(1, len(poly) // 150)].clip(0, 1).reshape(-1)
                lines.append(f"{remap[c]} " + " ".join(f"{v:.5f}" for v in pts))
            if not lines:
                continue
            ip = Path(r.path)
            key = f"{ip.parent.parent.name}__{ip.stem}"          # split__stem
            (pdir / f"{key}.txt").write_text("\n".join(lines) + "\n")
            cv2.imwrite(str(vdir / f"{key}.jpg"), r.plot())
            n += 1
    print(f"  {tag}: {n} images with proposals")
    return n


def apply_reviews(review_dir, manual):
    """Add approved proposals to the labels. Manual: a proposal counts only if its preview still exists."""
    added = rejected = 0
    for p in sorted((Path(review_dir) / "proposals").rglob("*.txt")):
        preview = Path(review_dir) / "previews" / p.parent.name / (p.stem + ".jpg")
        if manual and not preview.exists():
            rejected += 1
            continue
        split, stem = p.stem.split("__", 1)
        lbl = DATA / split / "labels" / f"{stem}.txt"
        old = lbl.read_text().rstrip("\n") if lbl.exists() else ""
        new = p.read_text().rstrip("\n")
        lbl.write_text((old + "\n" if old else "") + new + "\n")
        added += 1
    for c in DATA.rglob("*.cache"):
        c.unlink()
    print(f"Applied {added} images, rejected {rejected}")
    return added, rejected


def train_stage(vdir, st, name, start_weights, epochs):
    """Train (or resume after a disconnect). Results go to <version>/runs/<name> on Drive."""
    from ultralytics import YOLO
    run_dir = Path(vdir) / "runs" / name
    best, last = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"
    if st.get(f"{name}_done"):
        print(f"{name}: already trained")
        return best
    for c in DATA.rglob("*.cache"):
        c.unlink()
    if last.exists():
        print(f"{name}: resuming from {last}")
        try:
            YOLO(str(last)).train(resume=True)
        except Exception as e:      # e.g. it had already finished before the disconnect
            print("  resume not possible:", e)
    else:
        print(f"{name}: training from {start_weights} for up to {epochs} epochs")
        YOLO(str(start_weights)).train(
            data=str(DATA / "data.yaml"), task="segment", epochs=epochs, patience=PATIENCE,
            imgsz=IMGSZ, batch=BATCH, workers=WORKERS, amp=True, cos_lr=True, close_mosaic=10,
            device=0, project=str(Path(vdir) / "runs"), name=name, exist_ok=True, plots=True, seed=0)
    if not best.exists():
        raise RuntimeError(f"{name}: no best.pt produced, check the log above")
    st[f"{name}_done"] = True
    save_state(vdir, st)
    return best


def finish(vdir, st, best, names, extra):
    from ultralytics import YOLO
    vdir = Path(vdir)
    final = vdir / f"{vdir.name}.pt"
    shutil.copy(best, final)
    run_dir = Path(best).parent.parent
    for f in ["results.png", "results.csv", "confusion_matrix_normalized.png", "MaskPR_curve.png"]:
        if (run_dir / f).exists():
            shutil.copy(run_dir / f, vdir / f)
    info = {"version": vdir.name, "classes": names, "created": st.get("created"), **extra}
    (vdir / "info.json").write_text(json.dumps(info, indent=1))
    print("\nFinal check on the validation set:")
    YOLO(str(final)).val(data=str(DATA / "data.yaml"), imgsz=IMGSZ, batch=16, device=0,
                         project=str(LOCAL / "val"), name=vdir.name, exist_ok=True)
    st["done"] = True
    save_state(vdir, st)
    print(f"\nDONE. Model: {final}")

In [ ]:
from ultralytics import YOLO
ROOT_P = Path(ROOT)
assert ROOT_P.is_dir(), f"Create the folder {ROOT} in Google Drive first"

# --- which Program 1 result to use ---
done = {n: d for n, d in version_dirs(ROOT_P / "custom").items() if load_state(d).get("done")}
if CUSTOM_VERSION == "latest":
    assert done, "No finished Program 1 version found in yolo/custom"
    CDIR = done[max(done)]
else:
    CDIR = ROOT_P / "custom" / CUSTOM_VERSION
    assert load_state(CDIR).get("done"), f"{CDIR} is not a finished Program 1 version"
cinfo = json.loads((CDIR / "info.json").read_text())
CUSTOM_CLASSES = cinfo["classes"]
CUSTOM_MODEL = CDIR / f"{CDIR.name}.pt"
print("Custom model:", CDIR.name, CUSTOM_CLASSES)

coco_names = set(YOLO(MODEL).names.values())
bad = [c for c in COCO_CLASSES if c not in coco_names]
assert not bad, f"Not COCO classes: {bad}"
clash = set(COCO_CLASSES) & set(CUSTOM_CLASSES)
assert not clash, f"Both custom and COCO: {clash} (rename the custom one or drop it from COCO_CLASSES)"
NAMES = CUSTOM_CLASSES + COCO_CLASSES

if PER_CLASS_TRAIN is None:
    print(f"\n{'custom class':<20}{'train':>8}{'val':>8}")
    for n, c in cinfo["image_counts"].items():
        print(f"{n:<20}{c['train']:>8}{c['val']:>8}")
    raise StopHere("Choose PER_CLASS_TRAIN in the SETTINGS cell (often close to your custom train counts), then Run all.")

VDIR = resolve_version(ROOT_P / "full", VERSION, len(NAMES))
VNAME = VDIR.name
REVIEW_DIR = VDIR / "review"
st = load_state(VDIR)
if not st:
    st = {"created": today(), "classes": NAMES, "custom_version": CDIR.name,
          "per_class_train": PER_CLASS_TRAIN, "per_class_val": PER_CLASS_VAL}
    save_state(VDIR, st)
elif st["classes"] != NAMES or st["custom_version"] != CDIR.name:
    raise StopHere(f"{VNAME} was started with {st['custom_version']} and classes {st['classes']}.\n"
                   f"Put the settings back, or delete/rename the unfinished folder {VDIR} in Drive.")
elif st["per_class_train"] != PER_CLASS_TRAIN:
    print(f"Note: this version was started with PER_CLASS_TRAIN={st['per_class_train']}; keeping that.")
print("Version:", VNAME)

if st.get("built") and not (DATA / "data.yaml").exists():      # after a disconnect
    print("Restoring dataset from Drive ...")
    unzip(VDIR / "data.zip", LOCAL)
    if (VDIR / "labels_reviewed.zip").exists():
        unzip(VDIR / "labels_reviewed.zip", LOCAL)

## 1. Build: your custom dataset + selected COCO images

In [ ]:
if not st.get("built"):
    from collections import defaultdict
    from concurrent.futures import ThreadPoolExecutor
    import urllib.request
    from ultralytics.data.converter import merge_multi_segment

    shutil.rmtree(LOCAL, ignore_errors=True)
    unzip(CDIR / "dataset.zip", LOCAL)
    if (CDIR / "labels_reviewed.zip").exists():
        unzip(CDIR / "labels_reviewed.zip", LOCAL)
    for c in DATA.rglob("*.cache"):
        c.unlink()

    coco = LOCAL / "coco"
    coco.mkdir(parents=True, exist_ok=True)
    run(["wget", "-q", "http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "-P", str(coco)])
    unzip(coco / "annotations_trainval2017.zip", coco)
    random.seed(0)

    def select_and_convert(split, ann_file, per_class):
        d = json.load(open(ann_file))
        cat2new = {c["id"]: NAMES.index(c["name"]) for c in d["categories"] if c["name"] in COCO_CLASSES}
        imgs = {i["id"]: i for i in d["images"]}
        anns, crowd = defaultdict(list), set()
        for a in d["annotations"]:
            if a["category_id"] in cat2new:
                (crowd.add(a["image_id"]) if a.get("iscrowd") else anns[a["image_id"]].append(a))
        by_cls = defaultdict(set)
        for iid, al in anns.items():
            if iid not in crowd:
                for a in al:
                    by_cls[cat2new[a["category_id"]]].add(iid)
        chosen = set()
        for c in sorted(by_cls, key=lambda c: len(by_cls[c])):      # rarest class first
            pool = sorted(by_cls[c]); random.shuffle(pool)
            have = sum(1 for i in pool if i in chosen)
            for i in pool:
                if have >= per_class:
                    break
                if i not in chosen:
                    chosen.add(i); have += 1
        jobs = []
        for iid in sorted(chosen):
            im = imgs[iid]; w, h = im["width"], im["height"]
            lines = []
            for a in anns[iid]:
                seg = a["segmentation"]
                if not isinstance(seg, list) or not seg:
                    continue
                s = np.concatenate(merge_multi_segment(seg), axis=0) if len(seg) > 1 else np.array(seg[0]).reshape(-1, 2)
                s = (s / np.array([w, h])).clip(0, 1).reshape(-1)
                if len(s) >= 6:
                    lines.append(f"{cat2new[a['category_id']]} " + " ".join(f"{v:.5f}" for v in s))
            if lines:
                stem = "coco_" + Path(im["file_name"]).stem
                (DATA / split / "labels" / f"{stem}.txt").write_text("\n".join(lines) + "\n")
                jobs.append((im["coco_url"], DATA / split / "images" / f"{stem}.jpg"))
        counts = {NAMES[c]: sum(1 for i in by_cls[c] if i in chosen) for c in sorted(by_cls)}
        print(f"COCO {split}: {len(jobs)} images | per class: {counts}")
        return jobs, counts

    jobs, train_counts = select_and_convert("train", coco / "annotations/instances_train2017.json", st["per_class_train"])
    j2, val_counts = select_and_convert("val", coco / "annotations/instances_val2017.json", st["per_class_val"])
    jobs += j2

    def fetch(job):
        url, dst = job
        for _ in range(4):
            try:
                urllib.request.urlretrieve(url, dst); return True
            except Exception:
                pass
        return False
    with ThreadPoolExecutor(32) as ex:
        ok = list(ex.map(fetch, jobs))
    for (_, dst), k in zip(jobs, ok):
        if not k:
            (dst.parent.parent / "labels" / f"{dst.stem}.txt").unlink(missing_ok=True)
    print(f"downloaded {sum(ok)}, failed {len(ok) - sum(ok)}")
    shutil.rmtree(coco)

    write_yaml(NAMES)
    print("Saving dataset to Drive (a few minutes) ...")
    zip_dirs(VDIR / "data.zip", ["data"], cwd=LOCAL)
    st.update(built=True, coco_train_counts=train_counts, coco_val_counts=val_counts)
    save_state(VDIR, st)
else:
    print("Dataset already built")
for s in ["train", "val", "test"]:
    print(s, len(images_in(s)), "images")

## 2. Cross-label both ways

In [ ]:
if not st.get("proposed"):
    shutil.rmtree(REVIEW_DIR, ignore_errors=True)
    name2idx = {n: i for i, n in YOLO(MODEL).names.items()}
    keep = [name2idx[n] for n in COCO_CLASSES]
    remap = {name2idx[n]: NAMES.index(n) for n in COCO_CLASSES}
    custom_imgs = [p for p in images_in() if not p.name.startswith("coco_")]
    n1 = propose(MODEL, custom_imgs, keep, remap, CONF_COCO_ON_CUSTOM, "coco_objects_in_custom", REVIEW_DIR)
    ck = list(range(len(CUSTOM_CLASSES)))
    n2 = propose(CUSTOM_MODEL, images_in(prefix="coco_"), ck, {i: i for i in ck},
                 CONF_CUSTOM_ON_COCO, "custom_objects_in_coco", REVIEW_DIR)
    st.update(proposed=True, n_proposals=n1 + n2)
    save_state(VDIR, st)
else:
    print(f"Proposals already made: {st['n_proposals']}")

## 3. Review and apply
With `REVIEW = "manual"` the run stops here the first time. In Google Drive open
`yolo/full/<version>/review/previews/` (`coco_objects_in_custom`, `custom_objects_in_coco`).
**Delete every preview that is wrong**, set `REVIEWED_VERSION` to the version name, and Run all.

In [ ]:
if not st.get("applied"):
    n = st.get("n_proposals", 0)
    if n and REVIEW == "manual" and REVIEWED_VERSION != VNAME:
        raise StopHere(f"""
REVIEW NEEDED: {n} images have proposed labels.
  1. In Google Drive open: {REVIEW_DIR / 'previews'}
  2. Delete every preview image whose new outlines are wrong (kept = approved).
  3. In the SETTINGS cell set:  REVIEWED_VERSION = "{VNAME}"
  4. Runtime -> Run all""")
    added = apply_reviews(REVIEW_DIR, manual=(REVIEW == "manual"))[0] if n else 0
    zip_dirs(VDIR / "labels_reviewed.zip",
             ["data/train/labels", "data/val/labels", "data/test/labels"], cwd=LOCAL)
    st.update(applied=True, n_applied=added)
    save_state(VDIR, st)
else:
    print(f"Already applied: {st.get('n_applied', 0)} images")

## 4. Train and save the final model

In [ ]:
if st.get("done"):
    print("This version is finished:", VDIR / f"{VNAME}.pt")
else:
    best = train_stage(VDIR, st, "train", MODEL, EPOCHS)
    finish(VDIR, st, best, NAMES, {"custom_version": CDIR.name,
                                   "coco_train_counts": st["coco_train_counts"]})